In [371]:
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

## Data

### PPI

In [372]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [373]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

100%|██████████| 13715404/13715404 [00:02<00:00, 4852914.84it/s]


Number of pairs missing their reverse: 0
All pairs have their reverse present.


In [374]:
protein_interaction

,protein1,protein2,combined_score,Translated_protein_1,Translated_protein_2
0,9606.ENSP00000000233,9606.ENSP00000356607,173,ARF5,RALGPS2
1,9606.ENSP00000000233,9606.ENSP00000427567,154,ARF5,FHDC1
2,9606.ENSP00000000233,9606.ENSP00000253413,151,ARF5,ATP6V1E1
3,9606.ENSP00000000233,9606.ENSP00000493357,471,ARF5,CYTH2
4,9606.ENSP00000000233,9606.ENSP00000324127,201,ARF5,PSD3
...,...,...,...,...,...
13715399,9606.ENSP00000501317,9606.ENSP00000475489,195,RFX7,MPHOSPH9
13715400,9606.ENSP00000501317,9606.ENSP00000370447,158,RFX7,VCX
13715401,9606.ENSP00000501317,9606.ENSP00000312272,226,RFX7,YPEL2
13715402,9606.ENSP00000501317,9606.ENSP00000402092,169,RFX7,SAMD3


### DrugBank

In [375]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [376]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [377]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

Found 17430 drugs in the DrugBank XML.


### Genetic results

In [378]:
### import data

### genes
hpv_positive_genes  = pd.read_csv('Results/CNV results/HPV positive CNV top genes.csv')
hpv_negative_genes = pd.read_csv('Results/CNV results/HPV negative CNV top genes.csv')

### drug candiates
hpv_positive_direct_drug_candidates = pd.read_csv('Results/CNV results/HPV Positive Top Direct Drug Candidates Aggregated.csv')
hpv_positive_indirect_drug_candidates = pd.read_csv('Results/CNV results/HPV Positive Top Indirect Drug Candidates Aggregated.csv')

hpv_negative_direct_drug_candidates = pd.read_csv('Results/CNV results/HPV Negative Top Direct Drug Candidates Aggregated.csv')
#### no direct drug candidates came from Deletions, only amplifications
hpv_negative_direct_drug_candidates['MUT_TYPE'] = 'AMPLIFICATION'
hpv_negative_indirect_drug_candidates = pd.read_csv('Results/CNV results/HPV Negative Top Indirect Drug Candidates Aggregated.csv')

### somatic mtuation
hpv_positive_som_genes = pd.read_csv('Results/SOM results/HPV positive top genes.csv')
hpv_positive_som_direct_drug_candidates = pd.read_csv('Results/SOM results/hpv_positive_som_top_direct_drug_candidates_agg.csv')
hpv_positive_som_direct_drug_candidates['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_indirect_drug_candidates = pd.read_csv('Results/SOM results/hpv_positive_som_top_indirect_drug_candidates_agg.csv')
hpv_positive_som_indirect_drug_candidates['MUT_TYPE'] = 'SOMATIC'

hpv_negative_som_genes = pd.read_csv('Results/SOM results/HPV negative top genes.csv')
hpv_negative_som_direct_drug_candidates = pd.read_csv('Results/SOM results/hpv_negative_som_top_direct_drug_candidates_agg.csv')
hpv_negative_som_direct_drug_candidates['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_indirect_drug_candidates = pd.read_csv('Results/SOM results/hpv_negative_som_top_indirect_drug_candidates_agg.csv')
hpv_negative_som_indirect_drug_candidates['MUT_TYPE'] = 'SOMATIC'

#### Literature results

In [379]:
extracted_target_df= pd.read_csv('Validation pipeline/Results/cleaned_extracted_targets_all_pub_after_2000_GPU_2b_gemma.csv')
extracted_target_df_combined = pd.read_csv('Validation pipeline/Results/cleaned_extracted_combined_targets_all_pub_after_2000_GPU_2b_gemma.csv')

In [380]:
### accumulate all genes available in drugbank or ppi
Drug_bank_genes = list(Drug_bank['gene'].values)
ppi_genes = list(protein_interaction['Translated_protein_1'].values)
all_ppi_drugbank = list(set(Drug_bank_genes + ppi_genes))

## HPV+

#### Genes

In [381]:
hpv_positive_som_genes

,Gene,Count,Cohort_Frequency,Normalized_Count,Normalized_Cohort_Frequency,P_Value,Adjusted_P_Value,Significant,Empirical_P_Value,Adjusted_Empirical_P_Value,frequency_percentage,mutation_score
0,PIK3CA,18,17,0.002182,0.002061,1.055322e-21,4.630755e-18,True,0.0,0.0,23.611111,0.051515
1,ZNF750,11,8,0.003502,0.002547,4.795352e-16,1.052100e-12,True,0.0,0.0,11.111111,0.038912
2,CCDC191,6,6,0.001148,0.001148,1.514273e-06,1.107438e-03,True,0.0,0.0,8.333333,0.009564
3,FAM135B,6,5,0.001229,0.001024,1.024534e-06,1.107438e-03,True,0.0,0.0,6.944444,0.008533
4,AGO4,4,4,0.001549,0.001549,2.791318e-05,1.034515e-02,True,0.0,0.0,5.555556,0.008603
5,PRPF6,4,4,0.001417,0.001417,3.933866e-05,1.327831e-02,True,0.0,0.0,5.555556,0.007872


In [382]:
### combine hpv positive somatic genes and cnv genes
hpv_positive_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_genes['gene_name'] = hpv_positive_som_genes['Gene']
hpv_positive_som_genes['q_value']= hpv_positive_som_genes['Adjusted_P_Value']
hpv_positive_som_genes['empirical_q_value'] = hpv_positive_som_genes['Adjusted_Empirical_P_Value']
hpv_positive_combined_genes = pd.concat([hpv_positive_genes, hpv_positive_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_positive_combined_genes = hpv_positive_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()
hpv_positive_combined_genes


,gene_name,MUT_TYPE,q_value,empirical_q_value
0,ACE2,DELETION,3.07824363509455e-08,0.0254037070312238
1,ACTL6A,AMPLIFICATION,4.9013845356499404e-54,0.0116131232142737
2,ADIPOQ,AMPLIFICATION,4.9013845356499404e-54,0.0116131232142737
3,AGO4,SOMATIC,0.0103451516326857,0.0
4,AHSG,AMPLIFICATION,4.9013845356499404e-54,0.0116131232142737
5,ANOS1,DELETION,3.07824363509455e-08,0.0254037070312238
6,BMX,DELETION,3.07824363509455e-08,0.0254037070312238
7,CCDC191,SOMATIC,0.0011074380273068,0.0
8,CLDN1,AMPLIFICATION,6.5911053664735535e-53,0.0116131232142737
9,CNKSR2,DELETION,3.07824363509455e-08,0.0254037070312238


In [383]:
### merge genes with number of articles, pubmed id from literature data
hpv_positive_genes_with_lit = pd.merge(hpv_positive_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_positive_genes_with_lit.drop(columns =['INDEX'], inplace = True)

In [384]:
hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES
8,CLDN1,AMPLIFICATION,6.5911053664735535e-53,0.0116131232142737,CLDN1,"15170668, 17091452",2.0
30,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499404e-54, 4.6307549428396854e-18","0.0116131232142737, 0.0",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
36,RFC4,AMPLIFICATION,4.9013845356499404e-54,0.0116131232142737,RFC4,16467079,1.0
41,SOX2,AMPLIFICATION,1.2920437544143296e-55,0.0116131232142737,SOX2,15942670,1.0
43,TLR7,DELETION,3.07824363509455e-08,0.0254037070312238,TLR7,17201162,1.0


In [385]:
hpv_positive_genes_with_lit = hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

In [386]:
hpv_positive_genes_with_lit.to_csv('Results/HPV positive gene results.csv')

In [387]:
### export to final results output
hpv_positive_genes_with_lit.to_csv('Results/Final Results/HPV Positive validated genes.csv')

#### Direct

In [388]:
### merge all hpv postive direct drug candidates
hpv_positive_final_direct = pd.concat([hpv_positive_direct_drug_candidates, hpv_positive_som_direct_drug_candidates])
### group by drug and comma seperate genes and mutation type
### columns: DRUG	GENE_TARGET	NUM_DIRECT_TARGETS_HIT	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	GENE_GISTIC	GENE_normalized_gistic_score	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_p_value	drug_hypergeom_fdr	drug_empirical_p_value	drug_empirical_fdr	MUT_TYPE	GENE_Cohort_Frequency	
# GENE_Normalized_Count	GENE_Normalized_Cohort_Frequency	GENE_SIGNIFICANT
hpv_positive_final_direct = hpv_positive_final_direct.groupby('DRUG').agg({'GENE_TARGET': lambda x: ', '.join(x),
                                               'MUT_TYPE': lambda x: ', '.join(x),
                                                  'NUM_DIRECT_TARGETS_HIT': 'first',
                                                    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
                                                    'PERCENTAGE_OF_TARGETS_HIT': 'first',
                                                    'ACTION': 'first',
                                                    'SPECIFIC_FUNCTION': 'first',
                                                    'drug_hypergeom_p_value': 'first',
                                                    'drug_hypergeom_fdr': 'first',
                                                    'drug_empirical_p_value': 'first',
                                                    'drug_empirical_fdr': 'first',
                                                    }).reset_index()

In [389]:
hpv_positive_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.168234e-04,3.433760e-02,0.00047,0.038252
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.168234e-04,3.433760e-02,0.00047,0.038252
2,Cladribine,POLA1,DELETION,1,12,8.333333,inhibitor,chromatin binding,9.203648e-05,2.973096e-02,0.00009,0.031226
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.168234e-04,3.433760e-02,0.00047,0.038252
4,Copper,"AHSG, KNG1","AMPLIFICATION, AMPLIFICATION",2,146,1.369863,None,cysteine-type endopeptidase inhibitor activity,8.218045e-15,2.566222e-11,0.00001,0.002230
5,Golotimod,TLR7,DELETION,1,5,20.000000,None,double-stranded RNA binding,1.605536e-05,6.539420e-03,0.00002,0.007807
6,NADH,NDUFB5,AMPLIFICATION,1,144,0.694444,None,NADH dehydrogenase (ubiquinone) activity,6.205177e-08,2.325204e-05,0.00001,0.002230
7,TG-100801,VEGFD,DELETION,1,8,12.500000,inhibitor,chemoattractant activity,4.038151e-05,1.576225e-02,0.00001,0.004258
8,Wortmannin,PIK3CA,SOMATIC,1,5,20.000000,None,1-phosphatidylinositol-3-kinase activity,5.225777e-06,8.159180e-04,0.00002,0.003022
9,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.000000,None,1-phosphatidylinositol-3-kinase activity,7.766063e-05,8.661009e-03,0.00010,0.010893


In [390]:
extracted_target_df_combined

,GENE,PMID,INDEX,NUMBER_OF_ARTICLES
0,000-2,"11302242, 11302242","2610, 2610",1
1,10,"12608845, 12768769, 14967420, 15193028, 180565...","15288, 17016, 27005, 29481, 57658, 61105",6
2,106PRE,18186293,58737,1
3,106R,18186293,58737,1
4,106RECR,18186293,58737,1
...,...,...,...,...
6072,ZP-V3,12464647,13458,1
6073,ZP-V4,12464647,13458,1
6074,ZYGOMA,15883929,36459,1
6075,ZYGOMATIC,"12775236, 17522494","17081, 53057",2


In [391]:
#### add columns to hpv_positive_final_direct for PMIds and NUMBER_OF_ARTICLES from extracted_target_df_combined
### ADD COLUMNS: PMIDs, NUMBER_OF_ARTICLES, gene
### combine based on gene target, and if any of the genes in GENE_TARGET are in extracted_target_df_combined, then add the PMIDs and NUMBER_OF_ARTICLES

hpv_positive_final_direct['PMID'] = ''
hpv_positive_final_direct['NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_direct['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_direct.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    gene_targets = list(set(gene_targets))
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    hpv_positive_final_direct.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(list(set(literature_gene_targets)))
    hpv_positive_final_direct.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_positive_final_direct.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

In [392]:
hpv_positive_final_direct[hpv_positive_final_direct['NUMBER_OF_ARTICLES'] > 0]

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00047,0.038252,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00047,0.038252,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00047,0.038252,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA
5,Golotimod,TLR7,DELETION,1,5,20.0,None,double-stranded RNA binding,0.000016,0.006539,0.00002,0.007807,17201162,1,TLR7
8,Wortmannin,PIK3CA,SOMATIC,1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000005,0.000816,0.00002,0.003022,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA
9,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000078,0.008661,0.00010,0.010893,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA


In [393]:
### ensure that only drugs with NUMBER_OF_ARTICLES > 0 are saved, so that they have literature support
hpv_positive_final_direct = hpv_positive_final_direct[hpv_positive_final_direct['NUMBER_OF_ARTICLES'] > 0]
### save results
hpv_positive_final_direct.to_csv('Results/HPV Positive direct results.csv')

#### Indirect

In [394]:
### merge all hpv positive indirect drug candidates
hpv_positive_final_indirect = pd.concat([hpv_positive_indirect_drug_candidates, hpv_positive_som_indirect_drug_candidates], ignore_index=True)
hpv_positive_final_indirect['ACTION'] = hpv_positive_final_indirect['ACTION'].fillna('UNKNOWN')
hpv_positive_final_indirect['SPECIFIC_FUNCTION'] = hpv_positive_final_indirect['SPECIFIC_FUNCTION'].fillna('UNKNOWN')
hpv_positive_final_indirect['drug_hypergeom_fdr'] = hpv_positive_final_indirect['drug_hypergeom_fdr'].fillna('UNKNOWN')
hpv_positive_final_indirect['drug_empirical_fdr'] = hpv_positive_final_indirect['drug_empirical_fdr'].fillna('UNKNOWN')
hpv_positive_final_indirect['MUT_TYPE'] = hpv_positive_final_indirect['MUT_TYPE'].fillna('UNKNOWN')

### aggregate/group by drug name
### columns: DRUG	CONNECTED_TO (risk gene)	
# Number of risk or immediate neighbor genes
# targeted	total_genes_targeted_in_drugbank	
# PERCENTAGE_OF_TARGETS_HIT
# Number of indirect genes connected to this risk gene	
# GENE_TARGET	GENE_Cohort_Frequency	
# GENE_Normalized_Count	GENE_Normalized_Cohort_Frequency	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr

hpv_positive_final_indirect = hpv_positive_final_indirect.groupby(['DRUG']).agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'Number of risk or immediate neighbor genes targeted': 'first',
    'total_genes_targeted_in_drugbank': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'first',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'MUT_TYPE': lambda x: ', '.join(x)
}).reset_index()
hpv_positive_final_indirect.sort_values(by = 'drug_empirical_fdr', ascending = True).head(25)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
22,bosutinib,"HCK, FGR, SRC, MAP2K1, ABL1, LYN",PIK3CA,6,11,54.545455,inhibitor,"ATP binding, actin filament binding",1.295998e-04,0.001703,SOMATIC
1,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001703,SOMATIC
74,"inositol 1,3,4,5-tetrakisphosphate","PDPK1, BTK, CYTH2, CYTH3, AKT1",PIK3CA,5,8,62.500000,inhibitor,3-phosphoinositide-dependent protein kinase ac...,3.097531e-04,0.001703,SOMATIC
159,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,4.977113e-07,0.001703,SOMATIC
158,xl820,"KDR, PDGFRB, KIT, PDGFRA",PIK3CA,4,4,100.000000,UNKNOWN,ATP binding,1.931716e-04,0.001703,SOMATIC
157,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,4,5,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.220718e-05,0.001703,SOMATIC
49,entrectinib,"ROS1, JAK2, ALK, NTRK1, NTRK3",PIK3CA,5,7,71.428571,inhibitor,"ATP binding, acetylcholine receptor binding",1.915567e-04,0.001703,SOMATIC
131,su-11652,"KDR, PDGFRB, FLT1, PDGFRA",PIK3CA,4,5,80.000000,inhibitor,ATP binding,8.159180e-04,0.001703,SOMATIC
31,chlorprothixene,"CHRM1,CHRM3, CHRM4, CHRM5,HTR1A, DRD2, DRD3, C...","SST,KNG1,GNB4",8,11,72.727273,antagonist,G protein-coupled acetylcholine receptor activity,4.881950e-03,0.002230,AMPLIFICATION
114,quetiapine,"CHRM1,CHRM3, CHRM4, CHRM5,HTR1D, HTR1A, HTR7, ...","SST,KNG1,GNB4",13,24,54.166667,antagonist,G protein-coupled acetylcholine receptor activity,2.105585e-03,0.002230,AMPLIFICATION


In [395]:
hpv_positive_final_indirect[hpv_positive_final_indirect['ACTION']!= 'UNKNOWN'].sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending = [ True, False ]).head(50)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
1,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001703,SOMATIC
131,su-11652,"KDR, PDGFRB, FLT1, PDGFRA",PIK3CA,4,5,80.000000,inhibitor,ATP binding,8.159180e-04,0.001703,SOMATIC
49,entrectinib,"ROS1, JAK2, ALK, NTRK1, NTRK3",PIK3CA,5,7,71.428571,inhibitor,"ATP binding, acetylcholine receptor binding",1.915567e-04,0.001703,SOMATIC
74,"inositol 1,3,4,5-tetrakisphosphate","PDPK1, BTK, CYTH2, CYTH3, AKT1",PIK3CA,5,8,62.500000,inhibitor,3-phosphoinositide-dependent protein kinase ac...,3.097531e-04,0.001703,SOMATIC
22,bosutinib,"HCK, FGR, SRC, MAP2K1, ABL1, LYN",PIK3CA,6,11,54.545455,inhibitor,"ATP binding, actin filament binding",1.295998e-04,0.001703,SOMATIC
23,brigatinib,"IGF1R, MET, ALK, ERBB4, INSR, EGFR, ABL1, ERBB...","PIK3CA,VEGFD, PIK3CA",9,9,100.000000,"inhibitor, binding, inhibitor, binding","ATP binding, amyloid-beta binding, actin filam...",1.590798e-05,0.002230,"DELETION,AMPLIFICATION, SOMATIC"
35,conestat alfa,"C1R, C1S,PLAT, F12, F2,KLKB1, F11","MASP1,HRG,KNG1",7,7,100.000000,inhibitor,calcium ion binding,4.544467e-04,0.002230,AMPLIFICATION
51,erdafitinib,"FGFR1, FGFR4, FGFR3, KDR, PDGFRB, KIT, CSF1R, ...","PIK3CA,ANOS1,VEGFD, PIK3CA",10,10,100.000000,"inhibitor, substrate, inhibitor, substrate","ATP binding, ATP binding",3.033567e-06,0.002230,"DELETION,AMPLIFICATION, SOMATIC"
68,human c1-esterase inhibitor,"C1R, C1S,PLAT, F12, F2,KLKB1, F11","MASP1,HRG,KNG1",7,7,100.000000,inhibitor,calcium ion binding,4.544467e-04,0.002230,AMPLIFICATION
76,lenvatinib,"FGFR1, FGFR4, FGFR3, KDR, FLT1, KIT, RET, PDGF...","PIK3CA,ANOS1,VEGFD, PIK3CA",10,10,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",3.033567e-06,0.002230,"DELETION,AMPLIFICATION, SOMATIC"


In [396]:
#### add columns to hpv_positive_final_direct for PMIds and NUMBER_OF_ARTICLES from extracted_target_df_combined
### ADD COLUMNS: PMIDs, NUMBER_OF_ARTICLES, gene
### combine based on gene target, and if any of the genes in GENE_TARGET are in extracted_target_df_combined, then add the PMIDs and NUMBER_OF_ARTICLES
hpv_positive_final_indirect['PMID'] = ''
hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_indirect['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_indirect.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_positive_final_indirect.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_positive_final_indirect.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_positive_final_indirect.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

### validate risk genes
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_positive_final_indirect['RISK_GENE_PMID'] = ''
hpv_positive_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_indirect['RISK_GENE_LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_indirect.iterrows():
    risk_genes = row['CONNECTED_TO (risk gene)'].split(',')
    risk_genes = [gene.strip() for gene in risk_genes]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in risk_genes:
        #print(gene)
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_positive_final_indirect.at[index, 'RISK_GENE_LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_positive_final_indirect.at[index, 'RISK_GENE_PMID'] = ', '.join(pmids_set)
    hpv_positive_final_indirect.at[index, 'RISK_GENE_NUMBER_OF_ARTICLES'] = number_of_articles

### ensure that only drugs with NUMBER_OF_ARTICLES > 0 for both drug targets and risk genes are saved, so that they have literature support
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] > 0]

In [397]:
hpv_positive_final_indirect.to_csv('Results/HPV Positive indirect results.csv', index=False)

In [398]:
hpv_positive_final_indirect

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
0,1-tert-butyl-3-(4-chloro-phenyl)-1h-pyrazolo[3...,"SRC, LCK, LYN",PIK3CA,3,3,100.000000,UNKNOWN,ATP binding,3.802145e-03,0.003022,SOMATIC,"17961551, 17252232",2,"LYN, SRC","18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
1,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001703,SOMATIC,"18204781, 15947106",2,"JAK3, JAK2","18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
3,abrocitinib,"JAK2, TYK2, JAK3, JAK1, JAK2, TYK2, JAK3, JAK1","PIK3CA, PIK3CA",4,4,100.000000,"inhibitor, inhibitor","acetylcholine receptor binding, ATP binding, a...",3.433760e-02,0.033903,"AMPLIFICATION, SOMATIC","18204781, 15947106",4,"JAK3, JAK2","18376308, 15700036, 17990317, 11836556, 11...",34,PIK3CA
6,afatinib,"ERBB4, EGFR, ERBB2",PIK3CA,3,3,100.000000,inhibitor,"ATP binding, actin filament binding",3.802145e-03,0.007901,SOMATIC,"17826531, 14603452, 15161032, 17250463, 170984...",325,"ERBB2, ERBB4, EGFR","18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
8,altiratinib,"MET, KDR, NTRK1, NTRK3",PIK3CA,4,5,80.000000,"inhibitor, antagonist",ATP binding,8.159180e-04,0.003022,SOMATIC,"15735049, 11216646, 17435680, 11279608, 117524...",15,"NTRK3, MET, NTRK1","18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,xl228,"IGF1R, SRC, ABL1",PIK3CA,3,4,75.000000,UNKNOWN,"ATP binding, actin filament binding",1.400202e-02,0.015789,SOMATIC,"15221937, 17961551",2,"SRC, IGF1R","18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
157,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,4,5,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.220718e-05,0.001703,SOMATIC,"15379322, 18089792, 15355912, 17342092, 183165...",8,MTOR,"18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
158,xl820,"KDR, PDGFRB, KIT, PDGFRA",PIK3CA,4,4,100.000000,UNKNOWN,ATP binding,1.931716e-04,0.001703,SOMATIC,"16342249, 17317803, 17253141, 17513510, 144996...",9,"KIT, PDGFRA","18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
159,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,4.977113e-07,0.001703,SOMATIC,"11329134, 16807070, 11953175, 15988378, 155838...",55,"FGFR3, FGFR1, RET","18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA


In [399]:
hpv_positive_final_indirect.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending=[True, False]).head(50)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
158,xl820,"KDR, PDGFRB, KIT, PDGFRA",PIK3CA,4,4,100.000000,UNKNOWN,ATP binding,1.931716e-04,0.001703,SOMATIC,"16342249, 17317803, 17253141, 17513510, 144996...",9,"KIT, PDGFRA","18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
159,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,4.977113e-07,0.001703,SOMATIC,"11329134, 16807070, 11953175, 15988378, 155838...",55,"FGFR3, FGFR1, RET","18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
1,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001703,SOMATIC,"18204781, 15947106",2,"JAK3, JAK2","18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
131,su-11652,"KDR, PDGFRB, FLT1, PDGFRA",PIK3CA,4,5,80.000000,inhibitor,ATP binding,8.159180e-04,0.001703,SOMATIC,"17513510, 14499691",2,PDGFRA,"18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
157,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,4,5,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.220718e-05,0.001703,SOMATIC,"15379322, 18089792, 15355912, 17342092, 183165...",8,MTOR,"18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
49,entrectinib,"ROS1, JAK2, ALK, NTRK1, NTRK3",PIK3CA,5,7,71.428571,inhibitor,"ATP binding, acetylcholine receptor binding",1.915567e-04,0.001703,SOMATIC,"11216646, 15214912, 11279608, 15602716, 114858...",10,"NTRK3, ALK, JAK2, NTRK1","18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
74,"inositol 1,3,4,5-tetrakisphosphate","PDPK1, BTK, CYTH2, CYTH3, AKT1",PIK3CA,5,8,62.500000,inhibitor,3-phosphoinositide-dependent protein kinase ac...,3.097531e-04,0.001703,SOMATIC,"15896313, 14581353, 17974918, 12532415, 167780...",6,AKT1,"18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
22,bosutinib,"HCK, FGR, SRC, MAP2K1, ABL1, LYN",PIK3CA,6,11,54.545455,inhibitor,"ATP binding, actin filament binding",1.295998e-04,0.001703,SOMATIC,"16224162, 17961551, 17252232",3,"HCK, LYN, SRC","18376308, 15700036, 17990317, 11836556, 11...",17,PIK3CA
23,brigatinib,"IGF1R, MET, ALK, ERBB4, INSR, EGFR, ABL1, ERBB...","PIK3CA,VEGFD, PIK3CA",9,9,100.000000,"inhibitor, binding, inhibitor, binding","ATP binding, amyloid-beta binding, actin filam...",1.590798e-05,0.002230,"DELETION,AMPLIFICATION, SOMATIC","17826531, 14603452, 15161032, 17250463, 170984...",725,"ERBB4, IGF1R, ALK, EGFR, MET, ERBB2","18376308, 15700036, 17990317, 11836556, 11...",34,PIK3CA
51,erdafitinib,"FGFR1, FGFR4, FGFR3, KDR, PDGFRB, KIT, CSF1R, ...","PIK3CA,ANOS1,VEGFD, PIK3CA",10,10,100.000000,"inhibitor, substrate, inhibitor, substrate","ATP binding, ATP binding",3.033567e-06,0.002230,"DELETION,AMPLIFICATION, SOMATIC","11329134, 16807070, 16342249, 11953175, 172531...",145,"KIT, RET, PDGFRA, FGFR2, FGFR4, FGFR1, FGFR3","18376308, 15700036, 17990317, 11836556, 11...",34,PIK3CA


#### overall

In [400]:
hpv_positive_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00047,0.038252,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00047,0.038252,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00047,0.038252,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA
5,Golotimod,TLR7,DELETION,1,5,20.0,None,double-stranded RNA binding,0.000016,0.006539,0.00002,0.007807,17201162,1,TLR7
8,Wortmannin,PIK3CA,SOMATIC,1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000005,0.000816,0.00002,0.003022,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA
9,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000078,0.008661,0.00010,0.010893,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA


In [401]:
'artenimol' in hpv_positive_final_indirect['DRUG'].values

True

In [402]:
### combined direct and indirect final results
### final columns: DRUG   GENE_TARGET CONNECTED_TO (risk gene)	NUM_DIRECT_TARGETS_HIT  
# Number of risk or immediate neighbor genes targeted	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr	
### column for Target description: direct, or indirect
hpv_positive_final_direct['Target_Description'] = 'Direct'
hpv_positive_final_indirect['Target_Description'] = 'Indirect'
hpv_positive_final_results = pd.concat([hpv_positive_final_direct, hpv_positive_final_indirect], ignore_index=True)
### replace any null with 'NA' in the hwole dataframe
hpv_positive_final_results = hpv_positive_final_results.fillna('NA')
hpv_positive_final_results = hpv_positive_final_results.groupby('DRUG').agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'NUM_DIRECT_TARGETS_HIT': 'first',
    'Number of risk or immediate neighbor genes targeted': 'first',
    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'max',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'Target_Description': lambda x: ', '.join(x)
}).reset_index()

/var/folders/5p/swntgnbj3fbfxkx02kt3fq980000gn/T/ipykernel_97049/4020300996.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hpv_positive_final_direct['Target_Description'] = 'Direct'


In [403]:
hpv_positive_final_results.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending = [ True, False ]).head(50)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),NUM_DIRECT_TARGETS_HIT,Number of risk or immediate neighbor genes targeted,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,Target_Description
88,xl820,"KDR, PDGFRB, KIT, PDGFRA",PIK3CA,NA,4.0,NA,100.000000,UNKNOWN,ATP binding,1.931716e-04,0.001703,Indirect
89,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,NA,6.0,NA,100.000000,UNKNOWN,ATP binding,4.977113e-07,0.001703,Indirect
1,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,NA,4.0,NA,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001703,Indirect
79,su-11652,"KDR, PDGFRB, FLT1, PDGFRA",PIK3CA,NA,4.0,NA,80.000000,inhibitor,ATP binding,8.159180e-04,0.001703,Indirect
87,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,NA,4.0,NA,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.220718e-05,0.001703,Indirect
30,entrectinib,"ROS1, JAK2, ALK, NTRK1, NTRK3",PIK3CA,NA,5.0,NA,71.428571,inhibitor,"ATP binding, acetylcholine receptor binding",1.915567e-04,0.001703,Indirect
42,"inositol 1,3,4,5-tetrakisphosphate","PDPK1, BTK, CYTH2, CYTH3, AKT1",PIK3CA,NA,5.0,NA,62.500000,inhibitor,3-phosphoinositide-dependent protein kinase ac...,3.097531e-04,0.001703,Indirect
18,bosutinib,"HCK, FGR, SRC, MAP2K1, ABL1, LYN",PIK3CA,NA,6.0,NA,54.545455,inhibitor,"ATP binding, actin filament binding",1.295998e-04,0.001703,Indirect
19,brigatinib,"IGF1R, MET, ALK, ERBB4, INSR, EGFR, ABL1, ERBB...","PIK3CA,VEGFD, PIK3CA",NA,9.0,NA,100.000000,"inhibitor, binding, inhibitor, binding","ATP binding, amyloid-beta binding, actin filam...",1.590798e-05,0.002230,Indirect
31,erdafitinib,"FGFR1, FGFR4, FGFR3, KDR, PDGFRB, KIT, CSF1R, ...","PIK3CA,ANOS1,VEGFD, PIK3CA",NA,10.0,NA,100.000000,"inhibitor, substrate, inhibitor, substrate","ATP binding, ATP binding",3.033567e-06,0.002230,Indirect


## HPV-

#### Genes

In [405]:
### combine hpv negative somatic genes and cnv genes
hpv_negative_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_genes['gene_name'] = hpv_negative_som_genes['Gene']
hpv_negative_som_genes['q_value']= hpv_negative_som_genes['Adjusted_P_Value']
hpv_negative_som_genes['empirical_q_value'] = hpv_negative_som_genes['Adjusted_Empirical_P_Value']
hpv_negative_combined_genes = pd.concat([hpv_negative_genes, hpv_negative_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_negative_combined_genes = hpv_negative_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()
# hpv_negative_combined_genes.sort_values(by='q_value')

In [406]:
### merge genes with number of articles, pubmed id from literature data
hpv_negative_gene_results_with_lit = pd.merge(hpv_negative_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_negative_gene_results_with_lit.drop(columns = ['INDEX'], inplace = True)

In [407]:
hpv_negative_gene_results_with_lit = hpv_negative_gene_results_with_lit[hpv_negative_gene_results_with_lit['NUMBER_OF_ARTICLES']>0]

In [408]:
hpv_negative_gene_results_with_lit.sort_values(by = ['empirical_q_value', 'q_value'], ascending=[True, True], inplace=True)

In [409]:
hpv_negative_gene_results_with_lit.drop(columns=['INDEX', 'GENE'], errors='ignore', inplace=True)

In [410]:
hpv_negative_gene_results_with_lit.to_csv('Results/HPV negative gene results.csv')

In [411]:
hpv_negative_gene_results_with_lit

,gene_name,MUT_TYPE,q_value,empirical_q_value,PMID,NUMBER_OF_ARTICLES
204,TP53,SOMATIC,0.0,0.0,"11390535, 11445847, 11445859, 11916556, 125898...",29.0
162,PDE3A,SOMATIC,1.2533369131876885e-07,0.0,15078486,1.0
108,MAGEC1,SOMATIC,1.4965702636072946e-13,0.0,16929165,1.0
53,EPHA2,SOMATIC,1.5615622650296632e-16,0.0,"12494475, 16309192, 18030354, 18425361, 18485799",5.0
32,COL1A2,SOMATIC,2.018380908180692e-07,0.0,18254958,1.0
81,HRAS,SOMATIC,2.2137739229868268e-21,0.0,16676365,1.0
177,REG1A,SOMATIC,3.5624299726954185e-13,0.0,12901795,1.0
105,LRP1B,SOMATIC,4.228839135032655e-46,0.0,"15172977, 16857411, 16918994",3.0
175,RAC1,SOMATIC,4.606038103918665e-05,0.0,"17234718, 17592548",2.0
10,BCL6,AMPLIFICATION,0.0,0.0038533077280929,"11224600, 11420458, 14685876, 17429099",4.0


In [412]:
### export top genes to ouput final tables
hpv_negative_gene_results_with_lit.to_csv('Results/Final Results/HPV Negative validated genes.csv')

#### Direct

In [414]:
### combine all hpv negative direct drug candidates
hpv_negative_final_direct = pd.concat([hpv_negative_direct_drug_candidates, hpv_negative_som_direct_drug_candidates])
### group by drug and comma seperate genes and mutation type
### columns: DRUG	GENE_TARGET	NUM_DIRECT_TARGETS_HIT	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	GENE_GISTIC	GENE_normalized_gistic_score	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_p_value      

hpv_negative_final_direct = hpv_negative_final_direct.groupby('DRUG').agg({'GENE_TARGET': lambda x: ', '.join(x),
                                               'MUT_TYPE': lambda x: ', '.join(x.unique()), ### unique mutation types per drug
                                                  'NUM_DIRECT_TARGETS_HIT': 'first',
                                                    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
                                                    'PERCENTAGE_OF_TARGETS_HIT': 'first',
                                                    'ACTION': 'first',
                                                    'SPECIFIC_FUNCTION': 'first',
                                                    'drug_hypergeom_p_value': 'first',
                                                    'drug_hypergeom_fdr': 'first',
                                                    'drug_empirical_p_value': 'first',
                                                    'drug_empirical_fdr': 'first',
                                                    }).reset_index()

In [415]:
### validate hpv negative direct drug candidates with literature data
hpv_negative_final_direct['PMID'] = ''
hpv_negative_final_direct['NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_direct['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_direct.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    gene_targets = list(set(gene_targets))
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_direct.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_direct.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_negative_final_direct.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

    

In [416]:
hpv_negative_final_direct[hpv_negative_final_direct['NUMBER_OF_ARTICLES'] > 0].sort_values(by='NUMBER_OF_ARTICLES', ascending=False)

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,6.224881e-07,0.000486,0.00001,0.004258,"17638058, 11916556, 16969480, 16682429, 173127...",29,TP53
12,Wortmannin,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00041,0.046275,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA
13,XL765,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00043,0.046301,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA
7,Fostamatinib,"MAP3K13, PRKCI, TNIK, EPHA2","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,1.720246e-09,0.000002,0.00001,0.002927,"17990328, 18030354, 16309192, 18425361, 124944...",6,"EPHA2, PRKCI"
4,Dasatinib,EPHA2,SOMATIC,1.0,23.0,4.347826,antagonist,ATP binding,1.350788e-06,0.000744,0.00001,0.004258,"18030354, 16309192, 18425361, 12494475, 18485799",5,EPHA2
11,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,5.157973e-08,0.000104,0.00001,0.004258,"18030354, 16309192, 18425361, 12494475, 18485799",5,EPHA2
2,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.278605e-07,0.000080,0.00001,0.002927,17990328,1,PRKCI


In [417]:
### ensure that only drugs with NUMBER_OF_ARTICLES > 0 are saved, so that they have literature support
hpv_negative_final_direct= hpv_negative_final_direct[hpv_negative_final_direct['NUMBER_OF_ARTICLES'] > 0]
### save results
hpv_negative_final_direct.to_csv('Results/HPV Negative direct results.csv')

#### Indirect

In [418]:
### combine all hpv negative indirect drug candidates
hpv_negative_final_indirect = pd.concat([hpv_negative_indirect_drug_candidates, hpv_negative_som_indirect_drug_candidates])
hpv_negative_final_indirect['ACTION'] = hpv_negative_final_indirect['ACTION'].fillna('UNKNOWN')
hpv_negative_final_indirect['SPECIFIC_FUNCTION'] = hpv_negative_final_indirect['SPECIFIC_FUNCTION'].fillna('UNKNOWN')
hpv_negative_final_indirect['drug_hypergeom_fdr'] = hpv_negative_final_indirect['drug_hypergeom_fdr'].fillna('UNKNOWN')
hpv_negative_final_indirect['drug_empirical_fdr'] = hpv_negative_final_indirect['drug_empirical_fdr'].fillna('UNKNOWN')
hpv_negative_final_indirect['MUT_TYPE'] = hpv_negative_final_indirect['MUT_TYPE'].fillna('UNKNOWN')

### group by drug and comma seperate genes and mutation type
### columns: DRUG	CONNECTED_TO (risk gene)
# Number of risk or immediate neighbor genes
# targeted	total_genes_targeted_in_drugbank  
# PERCENTAGE_OF_TARGETS_HIT
# Number of indirect genes connected to this risk gene  
# GENE_TARGET	
# GENE_Cohort_Frequency   
# GENE_Normalized_Count	
# GENE_Normalized_Cohort_Frequency

hpv_negative_final_indirect = hpv_negative_final_indirect.groupby (['DRUG']).agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'Number of risk or immediate neighbor genes targeted': 'first',
    'total_genes_targeted_in_drugbank': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'first',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'MUT_TYPE': lambda x: ', '.join(x)
}).reset_index()

In [419]:
'artenimol' in hpv_negative_final_indirect['DRUG']

False

In [420]:
hpv_negative_final_indirect.sort_values(by = 'drug_empirical_fdr', ascending = True).head(20)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
52,nadh,"NDUFA8, NDUFS5, NDUFS2, NDUFA2, NDUFB10, NDUFV...",NDUFB5,58,144,40.277778,binder,"NADH dehydrogenase (ubiquinone) activity, 4 ir...",3.648098e-05,0.002927,AMPLIFICATION
85,sunitinib,"MET, KDR, PDGFRB, FLT1, KIT, CSF1R, PDGFRA, FL...","PIK3CA, PIK3CA",9,9,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",6.853670e-04,0.004258,"AMPLIFICATION, SOMATIC"
26,erdafitinib,"FGFR1, FGFR4, FGFR3, KDR, PDGFRB, KIT, CSF1R, ...","PIK3CA, PIK3CA",10,10,100.000000,"inhibitor, substrate, inhibitor, substrate","ATP binding, ATP binding",2.393902e-04,0.004258,"AMPLIFICATION, SOMATIC"
27,ethanol,"ACHE, GABRA2, GABRB2, GABRB1, GABRA1, GABRD, G...","SI, GABRG1, HTR1E, HTR5A, SLC17A6, CATSPERD, K...",28,54,51.851852,"activator, agonist, UNKNOWN, UNKNOWN, UNKNOWN,...","acetylcholine binding, benzodiazepine receptor...",2.596593e-04,0.004258,"SOMATIC, SOMATIC, SOMATIC, SOMATIC, SOMATIC, S..."
31,fludiazepam,"GABRA2, GABRB2, GABRB1, GABRA1, GABRD, GABRB3,...","GABRG1, GABRQ",13,17,76.470588,"potentiator, potentiator","benzodiazepine receptor activity, chloride cha...",1.260609e-03,0.004258,"SOMATIC, SOMATIC"
35,gamma-aminobutyric acid,"GABRA2, GABRB2, GABRA1, GABRA3, GABRG2, GABRA5...","GABRG1, GLUD2, SLC17A6, COL1A2",10,12,83.333333,"inhibitor, UNKNOWN, inhibitor, binder","benzodiazepine receptor activity, chloride cha...",4.233259e-03,0.004258,"SOMATIC, SOMATIC, SOMATIC, SOMATIC"
36,glutamic acid,"SLC7A11, GLS2, GLUD2, GLUD1, ALDH18A1, GPT2, G...","TP53, PDHA2, HTR5A, GLUD2, SLC17A6, GRM3, WARS...",35,61,57.377049,"UNKNOWN, substrate, UNKNOWN, substrate, UNKNOW...","cystine, glutaminase activity, ADP binding, AT...",2.751901e-07,0.004258,"SOMATIC, SOMATIC, SOMATIC, SOMATIC, SOMATIC, S..."
83,sorafenib,"RAF1, FGFR1, KDR, PDGFRB, EGFR, FLT1, BRAF, KI...","PIK3CA, PIK3CA",11,11,100.000000,"inhibitor, antagonist, inhibitor, antagonist","ATP binding, actin filament binding, ATP bindi...",1.044296e-04,0.004258,"AMPLIFICATION, SOMATIC"
44,lenvatinib,"FGFR1, FGFR4, FGFR3, KDR, FLT1, KIT, RET, PDGF...","PIK3CA, PIK3CA",10,10,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",2.393902e-04,0.004258,"AMPLIFICATION, SOMATIC"
46,lucitanib,"FGFR1, FGFR3, KDR, PDGFRB, FLT1, PDGFRA, FGFR2...","PIK3CA, PIK3CA",8,8,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",2.389931e-03,0.004258,"AMPLIFICATION, SOMATIC"


In [421]:
### add in literature validation columns
hpv_negative_final_indirect['PMID'] = ''
hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_indirect['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_indirect.iterrows():
    gene_targets = row['GENE_TARGET'].split(',')
    gene_targets = [gene.strip() for gene in gene_targets]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_indirect.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_indirect.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_negative_final_indirect.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles


### validate risk genes
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_negative_final_indirect['RISK_GENE_PMID'] = ''
hpv_negative_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_indirect['RISK_GENE_LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_indirect.iterrows():
    risk_genes = row['CONNECTED_TO (risk gene)'].split(',')
    risk_genes = [gene.strip() for gene in risk_genes]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in risk_genes:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_indirect.at[index, 'RISK_GENE_LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_indirect.at[index, 'RISK_GENE_PMID'] = ', '.join(pmids_set)
    hpv_negative_final_indirect.at[index, 'RISK_GENE_NUMBER_OF_ARTICLES'] = number_of_articles

### make sure only drugs with literature support for both drug targets and risk genes are saved
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] > 0]

hpv_negative_final_indirect.to_csv('Results/HPV Negative indirect results.csv', index=False)



In [422]:
hpv_negative_final_indirect.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending=[True, False]).head(50)[['DRUG', 'drug_empirical_fdr','LITERATURE_GENE_TARGETS', 'RISK_GENE_PMID','RISK_GENE_LITERATURE_GENE_TARGETS','RISK_GENE_NUMBER_OF_ARTICLES','MUT_TYPE']]

,DRUG,drug_empirical_fdr,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_LITERATURE_GENE_TARGETS,RISK_GENE_NUMBER_OF_ARTICLES,MUT_TYPE
15,brigatinib,0.004258,"ERBB4, IGF1R, ALK, EGFR, MET, ERBB2","15495191, 18413807, 16807070, 11309301, 168496...","PIK3CA, CDKN2A",50,"DELETION,AMPLIFICATION, SOMATIC"
26,erdafitinib,0.004258,"KIT, RET, PDGFRA, FGFR2, FGFR4, FGFR1, FGFR3","16380997, 11358835, 14581353, 15700036, 168070...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
44,lenvatinib,0.004258,"KIT, RET, PDGFRA, FGFR2, FGFR4, FGFR1, FGFR3","16380997, 11358835, 14581353, 15700036, 168070...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
46,lucitanib,0.004258,"FGFR3, FGFR1, PDGFRA, FGFR2","16380997, 11358835, 14581353, 15700036, 168070...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
55,nintedanib,0.004258,"LYN, PDGFRA, FGFR2, SRC, FGFR1, FGFR3","16380997, 11358835, 14581353, 15700036, 168070...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
58,pd-166326,0.004258,"KIT, PDGFRA, EGFR, SRC, FGFR1","15495191, 18413807, 11916556, 16807070, 176380...","PIK3CA, CDKN2A, TP53",79,"DELETION,AMPLIFICATION, SOMATIC, SOMATIC"
83,sorafenib,0.004258,"KIT, RET, EGFR, FGFR1, BRAF","16380997, 11358835, 14581353, 15700036, 168070...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
85,sunitinib,0.004258,"KIT, PDGFRA, MET","16380997, 11358835, 14581353, 15700036, 168070...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
64,ponatinib,0.004258,"KIT, RET, PDGFRA, LYN, FGFR2, SRC, FGFR4, FGFR...","15495191, 18413807, 16807070, 11309301, 168496...","PIK3CA, CDKN2A",50,"DELETION,AMPLIFICATION, SOMATIC"
66,pralsetinib,0.004258,"NTRK3, RET, FGFR2, JAK2, FGFR1, NTRK1","16380997, 11358835, 14581353, 15700036, 168070...",PIK3CA,34,"AMPLIFICATION, SOMATIC"


#### overall

In [423]:
hpv_negative_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,6.224881e-07,0.000486,0.00001,0.004258,"17638058, 11916556, 16969480, 16682429, 173127...",29,TP53
2,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.278605e-07,0.000080,0.00001,0.002927,17990328,1,PRKCI
4,Dasatinib,EPHA2,SOMATIC,1.0,23.0,4.347826,antagonist,ATP binding,1.350788e-06,0.000744,0.00001,0.004258,"18030354, 16309192, 18425361, 12494475, 18485799",5,EPHA2
7,Fostamatinib,"MAP3K13, PRKCI, TNIK, EPHA2","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,1.720246e-09,0.000002,0.00001,0.002927,"17990328, 18030354, 16309192, 18425361, 124944...",6,"EPHA2, PRKCI"
11,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,5.157973e-08,0.000104,0.00001,0.004258,"18030354, 16309192, 18425361, 12494475, 18485799",5,EPHA2
12,Wortmannin,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00041,0.046275,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA
13,XL765,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00043,0.046301,"16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA


In [424]:
hpv_negative_final_indirect

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
1,acetylsalicylic acid,"PRKAA1, PCNA, CASP3, CASP1, IKBKB, NFKBIA, PTG...","TP53, HRAS, PIK3CA, HAS2, UGT2B4",15,19,78.947368,"activator, downregulator, inhibitor, inhibitor...",[hydroxymethylglutaryl-CoA reductase (NADPH)] ...,0.000486,0.004258,"SOMATIC, SOMATIC, SOMATIC, SOMATIC, SOMATIC","17638058, 11854069, 12854173, 11173150, 110706...",114,"AKR1C1, TNFAIP6, MYC, PTGS2, TP53, NFKBIA, CCN...","17638058, 11916556, 16807070, 16969480, 178483...",47,"PIK3CA, HRAS, TP53"
3,ag-24322,"CDK1,CDK2, CDK4","CDKN2A,CDKN2B",3,3,100.000000,inhibitor,ATP binding,0.010932,0.018736,DELETION,"16525614, 12017338, 11267947, 11453659, 118540...",29,"CDK2, CDK4, CDK1","15495191, 16713672, 16278815, 12632081, 184138...",18,"CDKN2B, CDKN2A"
4,alsterpaullone,"CDK1, CDK5,CDK2","CDKN2A,CDKN2B",3,4,75.000000,inhibitor,"ATP binding, acetylcholine receptor activator ...",0.034197,0.030111,DELETION,"16525614, 17145757, 11267947, 11453659, 156454...",16,"CDK5, CDK1, CDK2","15495191, 16713672, 16278815, 12632081, 184138...",18,"CDKN2B, CDKN2A"
5,altiratinib,"MET, KDR, NTRK1, NTRK3,TEK","PIK3CA,THPO",5,5,100.000000,"inhibitor, antagonist",ATP binding,0.046272,0.046301,AMPLIFICATION,"15735049, 11216646, 17435680, 11279608, 117524...",15,"NTRK3, MET, NTRK1","16380997, 11358835, 14581353, 15700036, 168070...",17,PIK3CA
6,alvocidib,"CDK2, CDK4, CDK6, CDK7, CDK6, CDK9, CDK5, CDK8...","CDKN2B, TP53, PIK3CA",6,12,50.000000,"inhibitor, inhibitor, UNKNOWN","ATP binding, ATP binding, 7SK snRNA binding, a...",0.030578,0.046840,"DELETION, SOMATIC, SOMATIC","11854069, 17826531, 14603452, 15161032, 172504...",366,"CDK4, CDK6, CDK1, EGFR, CDK5, CDK2","17673925, 11916556, 17638058, 16807070, 169694...",48,"PIK3CA, CDKN2B, TP53"
7,amuvatinib,"MET, KIT, RET, PDGFRA, FLT3, RAD51, MET, KIT, ...","PIK3CA, TP53, PIK3CA",6,6,100.000000,"modulator, UNKNOWN, modulator","ATP binding, ATP binding, ATP binding",0.028340,0.028824,"AMPLIFICATION, SOMATIC, SOMATIC","11329134, 16342249, 11953175, 17253141, 159883...",136,"RAD51, KIT, RET, PDGFRA, MET","17638058, 11916556, 16807070, 16969480, 178483...",63,"PIK3CA, TP53"
9,arsenic trioxide,"MAPK1, JUN, AKT1, HDAC1,CCND1, CDKN1A, HDAC1, ...","CDKN2A,CDKN2B, TP53, PIK3CA, NFE2L3",6,10,60.000000,"inducer, inducer, antagonist, inducer, inducer","ATP binding, cAMP response element binding, 14...",0.003831,0.006939,"DELETION, SOMATIC, SOMATIC, SOMATIC","15896313, 18398822, 16807070, 16446704, 125324...",67,"HDAC1, AKT1, PML, CCND1, CDKN1A","15495191, 18413807, 17673925, 11916556, 176380...",64,"PIK3CA, CDKN2B, CDKN2A, TP53"
10,artenimol,"HNRNPK, DDX5,NPM1,ACTG1,ATP5F1A, ATP5MG, ATP5P...","FXR1,PIK3CA,ACTL6A,NDUFB5,SOX2,TNFSF10,P3H2,EI...",40,104,38.461538,"ligand, ligand, ligand, ligand, ligand, ligand...","cadherin binding, ATP binding, cadherin bindin...",0.014303,0.018131,"DELETION,AMPLIFICATION, SOMATIC, SOMATIC, SOMA...","17540725, 15381369, 15311506, 17565818, 163167...",24,"RPL14, GAPDH, PRDX1, HSPA8, LGALS1, VIM, RPS6,...","15495191, 18413807, 11916556, 16807070, 176380...",85,"PIK3CA, HRAS, SOX2, EIF4G1, CDKN2A, TP53, RAC1..."
13,bisindolylmaleimide i,"PDPK1, PRKCZ, PRKCI, LCK, AKT1,PRKCG, MAPK11, ...","PIK3CA,GNB4,SOX2,ECT2,DVL3,RFC4,MECOM,CDKN2A, ...",15,19,78.947368,"inhibitor, inhibitor, inhibitor, inhibitor",3-phosphoinositide-dependent protein kinase ac...,0.000239,0.004258,"DELETION,AMPLIFICATION, SOMATIC, SOMATIC, SOMATIC","16525614, 15896313, 14581353, 15645429, 168886...",35,"CDK1, AKT1, CHEK1, PRKCI, MAPK8","15495191, 18413807, 11916556, 16807070, 176380...",83,"PIK3CA, RFC4, SOX2, DVL3, CDKN2A, TP53"
14,bms-690514,"ERBB2, EGFR, FLT3",CDKN2A,3,4,75.000000,inhibitor,"ATP binding, 